# Assignment 2 — Dress Classification with DINOv2 Features and an MLP

**DS407 — Practical Deep Learning with Visual Data**

This notebook contains:

1. **Part I:** Data loading, label encoding, train/validation splitting, custom PyTorch Dataset, aspect-ratio-preserving resizing and padding, ImageNet normalization, DataLoaders, and visualization.
2. **Part II-A:** DINOv2 feature extraction for training, validation, and test images.
3. **MLP training:** Early stopping, learning-rate comparison, architecture comparison, batch-size comparison, scheduler comparison, final model selection, and test predictions.

The notebook is designed to run locally for testing and on a university cluster GPU for the complete experiment.

## Conceptual workflow

```text
Raw dress image
    ↓
Resize longest side to 224 while preserving aspect ratio
    ↓
Pad shorter side to obtain exactly 224 × 224
    ↓
Convert to tensor and apply ImageNet normalization
    ↓
Frozen DINOv2 encoder
    ↓
384-dimensional image embedding
    ↓
Trainable MLP classifier
    ↓
Predicted garment category
```

DINOv2 is used only as a **frozen feature extractor**. The MLP is the model trained for the dress-classification task.

In [ ]:
# ============================================================
# 1. IMPORTS, REPRODUCIBILITY, AND DIRECTORIES
# ============================================================

from pathlib import Path
import copy
import json
import os
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torchvision import transforms
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode


SEED = 42


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)

PROJECT_DIR = Path.cwd()

TRAIN_CSV = PROJECT_DIR / "train_2025.csv"
TEST_CSV = PROJECT_DIR / "test_2025.csv"

# The extracted image files should be in this directory.
IMAGE_DIR = PROJECT_DIR / "raw"

FEATURE_DIR = PROJECT_DIR / "features"
RESULTS_DIR = PROJECT_DIR / "results"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"

for directory in [FEATURE_DIR, RESULTS_DIR, CHECKPOINT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Store downloaded PyTorch models in a persistent cache.
os.environ.setdefault(
    "TORCH_HOME",
    str(Path.home() / ".cache" / "torch")
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Project directory:", PROJECT_DIR)
print("Device:", device)
print("PyTorch version:", torch.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Part I — Data preparation and understanding

The model input is the **dress image**. The target is the text category in `garment_types`.

Other text columns such as brand, colour, fabric, and neckline are not encoded because they are not used as model inputs in this assignment. The CSV mainly connects each `article_id` to the corresponding image and target category.

In [ ]:
# ============================================================
# 2. LOAD AND INSPECT THE CSV FILES
# ============================================================

if not TRAIN_CSV.exists():
    raise FileNotFoundError(f"Training CSV not found: {TRAIN_CSV}")

if not TEST_CSV.exists():
    raise FileNotFoundError(f"Test CSV not found: {TEST_CSV}")

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nTraining columns:")
print(train_df.columns.tolist())
print("\nFirst training rows:")
display(train_df.head())

In [ ]:
# ============================================================
# 3. BASIC DATA CHECKS
# ============================================================

IMAGE_COLUMN = "article_id"
CATEGORY_COLUMN = "garment_types"
LABEL_COLUMN = "label"
IMAGE_EXTENSION = ".jpg"

required_train_columns = {IMAGE_COLUMN, CATEGORY_COLUMN}
missing_train_columns = required_train_columns.difference(train_df.columns)

if missing_train_columns:
    raise ValueError(
        f"Missing training columns: {missing_train_columns}"
    )

if IMAGE_COLUMN not in test_df.columns:
    raise ValueError(
        f"Test CSV must contain '{IMAGE_COLUMN}'."
    )

print("Number of missing target labels:",
      train_df[CATEGORY_COLUMN].isna().sum())

class_counts = (
    train_df[CATEGORY_COLUMN]
    .value_counts()
    .sort_values(ascending=False)
)

print("\nClass distribution:")
display(class_counts.to_frame("number_of_images"))

plt.figure(figsize=(10, 5))
class_counts.plot(kind="bar")
plt.xlabel("Garment category")
plt.ylabel("Number of images")
plt.title("Training-set class distribution")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(
    RESULTS_DIR / "class_distribution.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# ============================================================
# 4. CREATE ONE CONSISTENT INTEGER-LABEL MAPPING
# ============================================================

class_names = sorted(
    train_df[CATEGORY_COLUMN]
    .dropna()
    .unique()
    .tolist()
)

class_to_idx = {
    class_name: class_index
    for class_index, class_name in enumerate(class_names)
}

idx_to_class = {
    class_index: class_name
    for class_name, class_index in class_to_idx.items()
}

train_df = train_df.copy()
train_df[LABEL_COLUMN] = (
    train_df[CATEGORY_COLUMN]
    .map(class_to_idx)
    .astype("int64")
)

NUM_CLASSES = len(class_names)

print("Number of classes:", NUM_CLASSES)
print("\nClass mapping:")

for class_name, class_index in class_to_idx.items():
    print(f"{class_index}: {class_name}")

In [ ]:
# ============================================================
# 5. REPRODUCIBLE STRATIFIED TRAIN/VALIDATION SPLIT
# ============================================================

train_split_df, val_split_df = train_test_split(
    train_df,
    test_size=0.20,
    random_state=SEED,
    stratify=train_df[LABEL_COLUMN]
)

train_split_df = train_split_df.reset_index(drop=True)
val_split_df = val_split_df.reset_index(drop=True)

TRAIN_SPLIT_CSV = PROJECT_DIR / "train_split.csv"
VAL_SPLIT_CSV = PROJECT_DIR / "validation_split.csv"

train_split_df.to_csv(TRAIN_SPLIT_CSV, index=False)
val_split_df.to_csv(VAL_SPLIT_CSV, index=False)

print("Training samples:", len(train_split_df))
print("Validation samples:", len(val_split_df))

train_proportions = (
    train_split_df[CATEGORY_COLUMN]
    .value_counts(normalize=True)
    .sort_index()
)

val_proportions = (
    val_split_df[CATEGORY_COLUMN]
    .value_counts(normalize=True)
    .sort_index()
)

distribution_comparison = pd.DataFrame({
    "train_proportion": train_proportions,
    "validation_proportion": val_proportions,
})

display(distribution_comparison)

## Correct image preprocessing

A direct `Resize((224, 224))` would stretch or squash dresses. Instead, the longest side is resized to 224 pixels and the shorter side is padded symmetrically.

The order is:

```text
RGB conversion
→ resize longest side to 224
→ pad to 224 × 224
→ ToTensor() maps pixel values from 0–255 to 0–1
→ ImageNet normalization
```

In [ ]:
# ============================================================
# 6. ASPECT-RATIO-PRESERVING RESIZE AND PADDING
# ============================================================

class ResizeLongestSideAndPad:
    """Resize the longest side and pad to a square image."""

    def __init__(self, target_size=224, fill=0):
        self.target_size = int(target_size)
        self.fill = fill

    def __call__(self, image):
        original_width, original_height = image.size

        if original_width <= 0 or original_height <= 0:
            raise ValueError(
                f"Invalid image dimensions: {image.size}"
            )

        scale = self.target_size / max(
            original_width,
            original_height
        )

        new_width = max(
            1,
            min(
                round(original_width * scale),
                self.target_size
            )
        )

        new_height = max(
            1,
            min(
                round(original_height * scale),
                self.target_size
            )
        )

        image = TF.resize(
            image,
            size=[new_height, new_width],
            interpolation=InterpolationMode.BILINEAR,
            antialias=True
        )

        total_horizontal_padding = (
            self.target_size - new_width
        )

        total_vertical_padding = (
            self.target_size - new_height
        )

        left = total_horizontal_padding // 2
        right = total_horizontal_padding - left

        top = total_vertical_padding // 2
        bottom = total_vertical_padding - top

        image = TF.pad(
            image,
            padding=[left, top, right, bottom],
            fill=self.fill,
            padding_mode="constant"
        )

        expected_size = (
            self.target_size,
            self.target_size
        )

        if image.size != expected_size:
            raise RuntimeError(
                f"Expected {expected_size}, obtained {image.size}."
            )

        return image

In [ ]:
# ============================================================
# 7. COMPLETE TRANSFORMATION PIPELINE
# ============================================================

TARGET_SIZE = 224

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

image_transform = transforms.Compose([
    ResizeLongestSideAndPad(
        target_size=TARGET_SIZE,
        fill=0
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

print(image_transform)

In [ ]:
# ============================================================
# 8. CUSTOM PYTORCH DATASET
# ============================================================

class ImageDressDataset(Dataset):
    """
    Load dress images using identifiers stored in a CSV file.

    Labelled mode returns:
        image_tensor, integer_label

    Unlabelled test mode returns:
        image_tensor, article_id
    """

    def __init__(
        self,
        csv_file,
        image_dir,
        image_column="article_id",
        label_column=None,
        transform=None,
        image_extension=".jpg"
    ):
        self.csv_file = Path(csv_file)
        self.image_dir = Path(image_dir)
        self.image_column = image_column
        self.label_column = label_column
        self.transform = transform
        self.image_extension = image_extension

        if not self.csv_file.exists():
            raise FileNotFoundError(
                f"CSV file not found: {self.csv_file}"
            )

        if not self.image_dir.exists():
            raise FileNotFoundError(
                f"Image directory not found: {self.image_dir}"
            )

        self.data = pd.read_csv(self.csv_file)

        required_columns = {self.image_column}

        if self.label_column is not None:
            required_columns.add(self.label_column)

        missing_columns = required_columns.difference(
            self.data.columns
        )

        if missing_columns:
            raise ValueError(
                f"Missing CSV columns: {missing_columns}. "
                f"Available columns: {self.data.columns.tolist()}"
            )

        self.identifiers = (
            self.data[self.image_column]
            .astype(str)
            .str.strip()
            .tolist()
        )

        if self.label_column is not None:
            self.labels = (
                self.data[self.label_column]
                .astype("int64")
                .tolist()
            )
        else:
            self.labels = None

    def __len__(self):
        return len(self.identifiers)

    def _build_image_path(self, article_id):
        article_path = Path(article_id)

        if article_path.suffix == "":
            article_path = article_path.with_suffix(
                self.image_extension
            )

        return self.image_dir / article_path

    def __getitem__(self, idx):
        article_id = self.identifiers[idx]
        image_path = self._build_image_path(article_id)

        if not image_path.exists():
            raise FileNotFoundError(
                f"Image not found: {image_path}"
            )

        with Image.open(image_path) as opened_image:
            image = opened_image.convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        if self.labels is None:
            return image, article_id

        return image, int(self.labels[idx])

In [ ]:
# ============================================================
# 9. CREATE DATASETS AND DATALOADERS
# ============================================================

# On Linux/cluster, several workers improve loading speed.
# On Windows/Jupyter, zero workers is the safest default.
NUM_WORKERS = 4 if os.name != "nt" else 0

# Safe for local testing; increase on the cluster if desired.
IMAGE_BATCH_SIZE = 64 if torch.cuda.is_available() else 16

train_dataset = ImageDressDataset(
    csv_file=TRAIN_SPLIT_CSV,
    image_dir=IMAGE_DIR,
    image_column=IMAGE_COLUMN,
    label_column=LABEL_COLUMN,
    transform=image_transform,
    image_extension=IMAGE_EXTENSION
)

val_dataset = ImageDressDataset(
    csv_file=VAL_SPLIT_CSV,
    image_dir=IMAGE_DIR,
    image_column=IMAGE_COLUMN,
    label_column=LABEL_COLUMN,
    transform=image_transform,
    image_extension=IMAGE_EXTENSION
)

test_dataset = ImageDressDataset(
    csv_file=TEST_CSV,
    image_dir=IMAGE_DIR,
    image_column=IMAGE_COLUMN,
    label_column=None,
    transform=image_transform,
    image_extension=IMAGE_EXTENSION
)

train_loader = DataLoader(
    train_dataset,
    batch_size=IMAGE_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
    persistent_workers=(NUM_WORKERS > 0)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=IMAGE_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
    persistent_workers=(NUM_WORKERS > 0)
)

test_loader = DataLoader(
    test_dataset,
    batch_size=IMAGE_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
    persistent_workers=(NUM_WORKERS > 0)
)

print("Image batch size:", IMAGE_BATCH_SIZE)
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
# ============================================================
# 10. VERIFY ONE SAMPLE AND ONE BATCH
# ============================================================

sample_image, sample_label = train_dataset[0]

print("One image shape:", sample_image.shape)
print("One integer label:", sample_label)
print("Category:", idx_to_class[sample_label])

assert sample_image.shape == (
    3,
    TARGET_SIZE,
    TARGET_SIZE
)

batch_images, batch_labels = next(
    iter(train_loader)
)

print("\nBatch image shape:", batch_images.shape)
print("Batch label shape:", batch_labels.shape)

assert batch_images.ndim == 4
assert batch_images.shape[1:] == (
    3,
    TARGET_SIZE,
    TARGET_SIZE
)
assert batch_labels.ndim == 1
assert batch_images.shape[0] == batch_labels.shape[0]

print("\nDataset and DataLoader checks passed.")

In [ ]:
# ============================================================
# 11. VISUALIZE TWO IMAGES FROM EACH TRAINING CATEGORY
# ============================================================

mean_tensor = torch.tensor(
    IMAGENET_MEAN
).view(3, 1, 1)

std_tensor = torch.tensor(
    IMAGENET_STD
).view(3, 1, 1)


def denormalize_image(image_tensor):
    image = image_tensor.cpu() * std_tensor + mean_tensor
    return image.clamp(0, 1)


indices_by_class = {}

for dataset_index, (_, label) in enumerate(train_dataset):
    indices_by_class.setdefault(label, [])

    if len(indices_by_class[label]) < 2:
        indices_by_class[label].append(dataset_index)

    if (
        len(indices_by_class) == NUM_CLASSES
        and all(
            len(indices) >= 2
            for indices in indices_by_class.values()
        )
    ):
        break


fig, axes = plt.subplots(
    nrows=NUM_CLASSES,
    ncols=2,
    figsize=(7, 3 * NUM_CLASSES)
)

for class_index in range(NUM_CLASSES):
    selected_indices = indices_by_class[class_index]

    for column_index, dataset_index in enumerate(
        selected_indices
    ):
        image_tensor, label = train_dataset[dataset_index]
        image = denormalize_image(
            image_tensor
        ).permute(1, 2, 0).numpy()

        axis = axes[class_index, column_index]
        axis.imshow(image)
        axis.set_title(idx_to_class[label])
        axis.axis("off")

plt.suptitle(
    "Two processed training images from each category",
    y=1.002
)
plt.tight_layout()
plt.savefig(
    RESULTS_DIR / "two_images_per_category.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

## Part II-A — DINOv2 feature extraction

The image DataLoaders from Part I now supply batches with shape:

```text
[batch_size, 3, 224, 224]
```

DINOv2 converts each image into one embedding. `dinov2_vits14` normally produces 384 features per image. The actual dimension is read directly from the extracted tensor rather than hard-coded.

The DINOv2 parameters remain frozen:

- no optimizer,
- no learning rate,
- no backward pass,
- no DINOv2 weight updates.

In [ ]:
# ============================================================
# 12. LOAD AND FREEZE PRETRAINED DINOV2
# ============================================================

print("Loading DINOv2...")

dinov2_model = torch.hub.load(
    "facebookresearch/dinov2",
    "dinov2_vits14"
)

dinov2_model = dinov2_model.to(device)
dinov2_model.eval()

for parameter in dinov2_model.parameters():
    parameter.requires_grad = False

print("DINOv2 loaded.")
print("All DINOv2 parameters frozen:",
      all(
          not parameter.requires_grad
          for parameter in dinov2_model.parameters()
      ))

In [ ]:
# ============================================================
# 13. TEST DINOV2 ON ONE IMAGE BATCH
# ============================================================

example_images, example_labels = next(
    iter(val_loader)
)

example_images = example_images.to(
    device,
    non_blocking=True
)

with torch.inference_mode():
    example_embeddings = dinov2_model(
        example_images
    )

print("Image batch shape:", example_images.shape)
print("Embedding batch shape:", example_embeddings.shape)

assert example_embeddings.ndim == 2
assert (
    example_embeddings.shape[0]
    == example_images.shape[0]
)

DINO_FEATURE_DIM = example_embeddings.shape[1]

print("DINOv2 feature dimension:", DINO_FEATURE_DIM)

del example_images
del example_embeddings

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# 14. FEATURE EXTRACTION FUNCTIONS
# ============================================================

def extract_labelled_features(
    model,
    data_loader,
    device,
    split_name
):
    model.eval()

    feature_batches = []
    label_batches = []

    start_time = time.time()

    with torch.inference_mode():
        for batch_index, (images, labels) in enumerate(
            data_loader,
            start=1
        ):
            images = images.to(
                device,
                non_blocking=True
            )

            embeddings = model(images)

            feature_batches.append(
                embeddings.cpu()
            )

            label_batches.append(
                labels.cpu().long()
            )

            if batch_index % 25 == 0:
                print(
                    f"{split_name}: "
                    f"{batch_index}/{len(data_loader)} batches"
                )

    features = torch.cat(
        feature_batches,
        dim=0
    )

    labels = torch.cat(
        label_batches,
        dim=0
    )

    elapsed_minutes = (
        time.time() - start_time
    ) / 60

    print(
        f"{split_name} completed in "
        f"{elapsed_minutes:.2f} minutes."
    )

    return features, labels


def extract_test_features(
    model,
    data_loader,
    device
):
    model.eval()

    feature_batches = []
    identifiers = []

    start_time = time.time()

    with torch.inference_mode():
        for batch_index, (
            images,
            batch_identifiers
        ) in enumerate(data_loader, start=1):
            images = images.to(
                device,
                non_blocking=True
            )

            embeddings = model(images)

            feature_batches.append(
                embeddings.cpu()
            )

            identifiers.extend(
                list(batch_identifiers)
            )

            if batch_index % 25 == 0:
                print(
                    "test: "
                    f"{batch_index}/{len(data_loader)} batches"
                )

    features = torch.cat(
        feature_batches,
        dim=0
    )

    elapsed_minutes = (
        time.time() - start_time
    ) / 60

    print(
        "test completed in "
        f"{elapsed_minutes:.2f} minutes."
    )

    return features, identifiers

In [ ]:
# ============================================================
# 15. DETERMINISTIC EXTRACTION LOADERS
# ============================================================

# No shuffling is used while saving extracted representations.
train_extraction_loader = DataLoader(
    train_dataset,
    batch_size=IMAGE_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
    persistent_workers=(NUM_WORKERS > 0)
)

val_extraction_loader = DataLoader(
    val_dataset,
    batch_size=IMAGE_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=False,
    persistent_workers=(NUM_WORKERS > 0)
)

In [ ]:
# ============================================================
# 16. EXTRACT TRAINING, VALIDATION, AND TEST FEATURES
# ============================================================

train_features, train_labels = extract_labelled_features(
    model=dinov2_model,
    data_loader=train_extraction_loader,
    device=device,
    split_name="train"
)

val_features, val_labels = extract_labelled_features(
    model=dinov2_model,
    data_loader=val_extraction_loader,
    device=device,
    split_name="validation"
)

test_features, test_identifiers = extract_test_features(
    model=dinov2_model,
    data_loader=test_loader,
    device=device
)

print("\nTraining features:", train_features.shape)
print("Training labels:", train_labels.shape)

print("\nValidation features:", val_features.shape)
print("Validation labels:", val_labels.shape)

print("\nTest features:", test_features.shape)
print("Test identifiers:", len(test_identifiers))

assert train_features.shape[0] == train_labels.shape[0]
assert val_features.shape[0] == val_labels.shape[0]
assert test_features.shape[0] == len(test_identifiers)

assert train_features.shape[1] == val_features.shape[1]
assert train_features.shape[1] == test_features.shape[1]

assert not torch.isnan(train_features).any()
assert not torch.isnan(val_features).any()
assert not torch.isnan(test_features).any()

DINO_FEATURE_DIM = train_features.shape[1]

print("\nVerified DINOv2 feature dimension:",
      DINO_FEATURE_DIM)

In [ ]:
# ============================================================
# 17. SAVE EXTRACTED FEATURES ONCE
# ============================================================

torch.save(
    {
        "features": train_features,
        "labels": train_labels,
        "class_names": class_names,
        "class_to_idx": class_to_idx,
        "feature_dimension": DINO_FEATURE_DIM
    },
    FEATURE_DIR / "train_dinov2_features.pt"
)

torch.save(
    {
        "features": val_features,
        "labels": val_labels,
        "class_names": class_names,
        "class_to_idx": class_to_idx,
        "feature_dimension": DINO_FEATURE_DIM
    },
    FEATURE_DIR / "validation_dinov2_features.pt"
)

torch.save(
    {
        "features": test_features,
        "identifiers": test_identifiers,
        "class_names": class_names,
        "class_to_idx": class_to_idx,
        "feature_dimension": DINO_FEATURE_DIM
    },
    FEATURE_DIR / "test_dinov2_features.pt"
)

print("Saved feature files:")

for path in sorted(FEATURE_DIR.glob("*.pt")):
    print(path.name, f"({path.stat().st_size / 1e6:.2f} MB)")

## MLP classifier

The MLP receives DINOv2 embeddings, not raw images.

Initial architecture:

```text
DINO feature dimension
→ Linear(256)
→ ReLU
→ Dropout
→ Linear(128)
→ ReLU
→ Dropout
→ Linear(number of dress classes)
```

The final layer returns raw logits. No Softmax is used inside the model because `CrossEntropyLoss` handles the required log-softmax operation internally.

In [ ]:
# ============================================================
# 18. FEATURE DATASETS AND LOADERS
# ============================================================

feature_train_dataset = TensorDataset(
    train_features.float(),
    train_labels.long()
)

feature_val_dataset = TensorDataset(
    val_features.float(),
    val_labels.long()
)


def create_feature_loaders(
    batch_size,
    seed=42
):
    generator = torch.Generator()
    generator.manual_seed(seed)

    train_feature_loader = DataLoader(
        feature_train_dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
        num_workers=0,
        drop_last=False
    )

    val_feature_loader = DataLoader(
        feature_val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        drop_last=False
    )

    return train_feature_loader, val_feature_loader

In [ ]:
# ============================================================
# 19. DEFINE THE MLP CLASSIFIER
# ============================================================

class MLPClassifier(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dims,
        num_classes,
        dropout=0.20
    ):
        super().__init__()

        layers = []
        previous_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(previous_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            previous_dim = hidden_dim

        layers.append(
            nn.Linear(previous_dim, num_classes)
        )

        self.network = nn.Sequential(*layers)

    def forward(self, features):
        return self.network(features)


initial_mlp = MLPClassifier(
    input_dim=DINO_FEATURE_DIM,
    hidden_dims=[256, 128],
    num_classes=NUM_CLASSES,
    dropout=0.20
)

print(initial_mlp)

example_output = initial_mlp(
    train_features[:8].float()
)

print("Example output shape:", example_output.shape)

assert example_output.shape == (
    8,
    NUM_CLASSES
)

In [ ]:
# ============================================================
# 20. TRAINING AND VALIDATION FUNCTIONS
# ============================================================

def train_mlp_one_epoch(
    model,
    data_loader,
    loss_function,
    optimizer,
    device
):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for features, labels in data_loader:
        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(features)
        loss = loss_function(logits, labels)

        loss.backward()
        optimizer.step()

        predictions = logits.argmax(dim=1)
        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_correct += (
            predictions == labels
        ).sum().item()
        total_samples += batch_size

    return (
        total_loss / total_samples,
        total_correct / total_samples
    )


def evaluate_mlp(
    model,
    data_loader,
    loss_function,
    device
):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.inference_mode():
        for features, labels in data_loader:
            features = features.to(device)
            labels = labels.to(device)

            logits = model(features)
            loss = loss_function(logits, labels)

            predictions = logits.argmax(dim=1)
            batch_size = labels.size(0)

            total_loss += loss.item() * batch_size
            total_correct += (
                predictions == labels
            ).sum().item()
            total_samples += batch_size

    return (
        total_loss / total_samples,
        total_correct / total_samples
    )

In [ ]:
# ============================================================
# 21. TRAINING LOOP WITH EARLY STOPPING
# ============================================================

def fit_mlp(
    model,
    train_loader,
    val_loader,
    learning_rate,
    max_epochs=50,
    patience=5,
    scheduler_name=None,
    checkpoint_path=None
):
    model = model.to(device)

    loss_function = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )

    scheduler = None

    if scheduler_name == "ReduceLROnPlateau":
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=2,
            min_lr=1e-6
        )

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
        "learning_rate": []
    }

    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())
    best_epoch = 0
    epochs_without_improvement = 0

    for epoch in range(1, max_epochs + 1):
        current_lr = optimizer.param_groups[0]["lr"]

        train_loss, train_accuracy = train_mlp_one_epoch(
            model,
            train_loader,
            loss_function,
            optimizer,
            device
        )

        val_loss, val_accuracy = evaluate_mlp(
            model,
            val_loader,
            loss_function,
            device
        )

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)
        history["learning_rate"].append(current_lr)

        print(
            f"Epoch {epoch:02d} | "
            f"LR {current_lr:.2e} | "
            f"train loss {train_loss:.4f} | "
            f"train acc {train_accuracy:.4f} | "
            f"val loss {val_loss:.4f} | "
            f"val acc {val_accuracy:.4f}"
        )

        if scheduler is not None:
            scheduler.step(val_loss)

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(
                model.state_dict()
            )
            epochs_without_improvement = 0

            if checkpoint_path is not None:
                torch.save(
                    best_state,
                    checkpoint_path
                )
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(
                "Early stopping: "
                f"no validation-loss improvement for "
                f"{patience} consecutive epochs."
            )
            break

    model.load_state_dict(best_state)

    return {
        "model": model.to("cpu"),
        "history": history,
        "best_val_loss": best_val_loss,
        "best_val_accuracy": max(
            history["val_accuracy"]
        ),
        "best_epoch": best_epoch
    }

In [ ]:
# ============================================================
# 22. HELPER FOR ONE REPRODUCIBLE EXPERIMENT
# ============================================================

def run_mlp_experiment(
    experiment_name,
    hidden_dims,
    learning_rate,
    batch_size,
    dropout=0.20,
    max_epochs=40,
    patience=5,
    scheduler_name=None
):
    print("\n" + "=" * 72)
    print("Experiment:", experiment_name)
    print("Hidden dimensions:", hidden_dims)
    print("Learning rate:", learning_rate)
    print("Batch size:", batch_size)
    print("Scheduler:", scheduler_name)
    print("=" * 72)

    set_seed(SEED)

    experiment_train_loader, experiment_val_loader = (
        create_feature_loaders(
            batch_size=batch_size,
            seed=SEED
        )
    )

    model = MLPClassifier(
        input_dim=DINO_FEATURE_DIM,
        hidden_dims=hidden_dims,
        num_classes=NUM_CLASSES,
        dropout=dropout
    )

    checkpoint_path = (
        CHECKPOINT_DIR /
        f"{experiment_name}.pt"
    )

    result = fit_mlp(
        model=model,
        train_loader=experiment_train_loader,
        val_loader=experiment_val_loader,
        learning_rate=learning_rate,
        max_epochs=max_epochs,
        patience=patience,
        scheduler_name=scheduler_name,
        checkpoint_path=checkpoint_path
    )

    result["experiment_name"] = experiment_name
    result["hidden_dims"] = hidden_dims
    result["learning_rate"] = learning_rate
    result["batch_size"] = batch_size
    result["dropout"] = dropout
    result["scheduler_name"] = scheduler_name

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

## Guided hyperparameter search

To limit computation, the experiments are conducted sequentially:

1. Fix architecture `[256]`, Adam, and batch size 64; compare learning rates.
2. With the best learning rate, compare MLP architectures.
3. With the best learning rate and architecture, compare batch sizes.
4. Apply one scheduler to the best constant-learning-rate configuration.

In [ ]:
# ============================================================
# 23. LEARNING-RATE SCAN
# ============================================================

LEARNING_RATES = [
    1e-2,
    5e-3,
    1e-3,
    5e-4,
    1e-4
]

lr_results = {}

for learning_rate in LEARNING_RATES:
    experiment_name = (
        f"lr_{learning_rate:.0e}"
        .replace("-", "minus")
    )

    lr_results[learning_rate] = run_mlp_experiment(
        experiment_name=experiment_name,
        hidden_dims=[256],
        learning_rate=learning_rate,
        batch_size=64,
        dropout=0.20,
        max_epochs=30,
        patience=4,
        scheduler_name=None
    )

In [ ]:
# ============================================================
# 24. PLOT LEARNING-RATE COMPARISON
# ============================================================

plt.figure(figsize=(9, 6))

for learning_rate, result in lr_results.items():
    history = result["history"]

    plt.plot(
        range(1, len(history["val_loss"]) + 1),
        history["val_loss"],
        marker="o",
        label=f"LR={learning_rate:.0e}"
    )

plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("MLP validation loss: learning-rate comparison")
plt.legend()
plt.tight_layout()
plt.savefig(
    RESULTS_DIR / "mlp_learning_rate_val_loss.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()


plt.figure(figsize=(9, 6))

for learning_rate, result in lr_results.items():
    history = result["history"]

    plt.plot(
        range(1, len(history["val_accuracy"]) + 1),
        history["val_accuracy"],
        marker="o",
        label=f"LR={learning_rate:.0e}"
    )

plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.title("MLP validation accuracy: learning-rate comparison")
plt.legend()
plt.tight_layout()
plt.savefig(
    RESULTS_DIR / "mlp_learning_rate_val_accuracy.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# ============================================================
# 25. SELECT BEST LEARNING RATE
# ============================================================

best_learning_rate = min(
    lr_results,
    key=lambda lr: (
        lr_results[lr]["best_val_loss"],
        -lr_results[lr]["best_val_accuracy"]
    )
)

print("Best learning rate:", best_learning_rate)

lr_summary = pd.DataFrame([
    {
        "learning_rate": learning_rate,
        "best_epoch": result["best_epoch"],
        "best_validation_loss": result["best_val_loss"],
        "best_validation_accuracy": result["best_val_accuracy"]
    }
    for learning_rate, result in lr_results.items()
]).sort_values(
    "best_validation_loss"
)

display(lr_summary)

lr_summary.to_csv(
    RESULTS_DIR / "learning_rate_results.csv",
    index=False
)

In [ ]:
# ============================================================
# 26. ARCHITECTURE COMPARISON
# ============================================================

ARCHITECTURES = {
    "128": [128],
    "256": [256],
    "512": [512],
    "256_128": [256, 128]
}

architecture_results = {}

for architecture_name, hidden_dims in ARCHITECTURES.items():
    architecture_results[architecture_name] = (
        run_mlp_experiment(
            experiment_name=f"architecture_{architecture_name}",
            hidden_dims=hidden_dims,
            learning_rate=best_learning_rate,
            batch_size=64,
            dropout=0.20,
            max_epochs=40,
            patience=5,
            scheduler_name=None
        )
    )

In [ ]:
# ============================================================
# 27. PLOT AND SELECT BEST ARCHITECTURE
# ============================================================

plt.figure(figsize=(9, 6))

for architecture_name, result in architecture_results.items():
    history = result["history"]

    plt.plot(
        range(1, len(history["val_loss"]) + 1),
        history["val_loss"],
        marker="o",
        label=architecture_name
    )

plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("MLP validation loss: architecture comparison")
plt.legend()
plt.tight_layout()
plt.savefig(
    RESULTS_DIR / "mlp_architecture_val_loss.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()


best_architecture_name = min(
    architecture_results,
    key=lambda name: (
        architecture_results[name]["best_val_loss"],
        -architecture_results[name]["best_val_accuracy"]
    )
)

best_hidden_dims = ARCHITECTURES[
    best_architecture_name
]

print("Best architecture:", best_architecture_name)
print("Hidden dimensions:", best_hidden_dims)

architecture_summary = pd.DataFrame([
    {
        "architecture": architecture_name,
        "hidden_dimensions": str(
            ARCHITECTURES[architecture_name]
        ),
        "best_epoch": result["best_epoch"],
        "best_validation_loss": result["best_val_loss"],
        "best_validation_accuracy": result["best_val_accuracy"]
    }
    for architecture_name, result
    in architecture_results.items()
]).sort_values(
    "best_validation_loss"
)

display(architecture_summary)

architecture_summary.to_csv(
    RESULTS_DIR / "architecture_results.csv",
    index=False
)

In [ ]:
# ============================================================
# 28. BATCH-SIZE COMPARISON
# ============================================================

BATCH_SIZES = [64, 128]

batch_size_results = {}

for feature_batch_size in BATCH_SIZES:
    batch_size_results[feature_batch_size] = (
        run_mlp_experiment(
            experiment_name=f"batch_{feature_batch_size}",
            hidden_dims=best_hidden_dims,
            learning_rate=best_learning_rate,
            batch_size=feature_batch_size,
            dropout=0.20,
            max_epochs=40,
            patience=5,
            scheduler_name=None
        )
    )

best_feature_batch_size = min(
    batch_size_results,
    key=lambda batch_size: (
        batch_size_results[batch_size]["best_val_loss"],
        -batch_size_results[batch_size]["best_val_accuracy"]
    )
)

batch_size_summary = pd.DataFrame([
    {
        "batch_size": batch_size,
        "best_epoch": result["best_epoch"],
        "best_validation_loss": result["best_val_loss"],
        "best_validation_accuracy": result["best_val_accuracy"]
    }
    for batch_size, result in batch_size_results.items()
]).sort_values(
    "best_validation_loss"
)

display(batch_size_summary)

print("Best feature batch size:",
      best_feature_batch_size)

batch_size_summary.to_csv(
    RESULTS_DIR / "batch_size_results.csv",
    index=False
)

In [ ]:
# ============================================================
# 29. SCHEDULER COMPARISON
# ============================================================

constant_result = run_mlp_experiment(
    experiment_name="final_constant_lr",
    hidden_dims=best_hidden_dims,
    learning_rate=best_learning_rate,
    batch_size=best_feature_batch_size,
    dropout=0.20,
    max_epochs=60,
    patience=7,
    scheduler_name=None
)

scheduler_result = run_mlp_experiment(
    experiment_name="final_reduce_on_plateau",
    hidden_dims=best_hidden_dims,
    learning_rate=best_learning_rate,
    batch_size=best_feature_batch_size,
    dropout=0.20,
    max_epochs=60,
    patience=7,
    scheduler_name="ReduceLROnPlateau"
)

In [ ]:
# ============================================================
# 30. PLOT SCHEDULER IMPACT
# ============================================================

plt.figure(figsize=(9, 6))

for label, result in {
    "Constant LR": constant_result,
    "ReduceLROnPlateau": scheduler_result
}.items():
    history = result["history"]

    plt.plot(
        range(1, len(history["val_loss"]) + 1),
        history["val_loss"],
        marker="o",
        label=label
    )

plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("Scheduler impact on validation loss")
plt.legend()
plt.tight_layout()
plt.savefig(
    RESULTS_DIR / "scheduler_val_loss.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()


plt.figure(figsize=(9, 6))

scheduler_history = scheduler_result["history"]

plt.plot(
    range(1, len(scheduler_history["learning_rate"]) + 1),
    scheduler_history["learning_rate"],
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("Learning rate")
plt.yscale("log")
plt.title("Learning-rate schedule")
plt.tight_layout()
plt.savefig(
    RESULTS_DIR / "scheduler_learning_rate.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# ============================================================
# 31. CHOOSE AND SAVE THE FINAL MLP
# ============================================================

candidate_final_results = {
    "Constant learning rate": constant_result,
    "ReduceLROnPlateau": scheduler_result
}

final_strategy = min(
    candidate_final_results,
    key=lambda name: (
        candidate_final_results[name]["best_val_loss"],
        -candidate_final_results[name]["best_val_accuracy"]
    )
)

final_result = candidate_final_results[
    final_strategy
]

final_model = final_result["model"]

FINAL_MODEL_PATH = (
    CHECKPOINT_DIR /
    "final_dinov2_mlp_classifier.pt"
)

torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "input_dimension": DINO_FEATURE_DIM,
        "hidden_dimensions": best_hidden_dims,
        "num_classes": NUM_CLASSES,
        "class_names": class_names,
        "class_to_idx": class_to_idx,
        "learning_rate": best_learning_rate,
        "batch_size": best_feature_batch_size,
        "dropout": 0.20,
        "optimizer": "Adam",
        "scheduler": final_strategy,
        "best_epoch": final_result["best_epoch"],
        "best_validation_loss": final_result["best_val_loss"],
        "best_validation_accuracy": final_result[
            "best_val_accuracy"
        ]
    },
    FINAL_MODEL_PATH
)

final_hyperparameters = pd.DataFrame([
    {
        "component": "DINOv2 model",
        "selected_value": "dinov2_vits14 (frozen)"
    },
    {
        "component": "DINOv2 feature dimension",
        "selected_value": DINO_FEATURE_DIM
    },
    {
        "component": "MLP hidden layers",
        "selected_value": str(best_hidden_dims)
    },
    {
        "component": "Activation",
        "selected_value": "ReLU"
    },
    {
        "component": "Dropout",
        "selected_value": 0.20
    },
    {
        "component": "Optimizer",
        "selected_value": "Adam"
    },
    {
        "component": "Learning rate",
        "selected_value": best_learning_rate
    },
    {
        "component": "Batch size",
        "selected_value": best_feature_batch_size
    },
    {
        "component": "Scheduler",
        "selected_value": final_strategy
    },
    {
        "component": "Early-stopping patience",
        "selected_value": 7
    },
    {
        "component": "Best validation loss",
        "selected_value": final_result["best_val_loss"]
    },
    {
        "component": "Best validation accuracy",
        "selected_value": final_result[
            "best_val_accuracy"
        ]
    }
])

display(final_hyperparameters)

final_hyperparameters.to_csv(
    RESULTS_DIR / "final_hyperparameters.csv",
    index=False
)

print("Final model saved to:", FINAL_MODEL_PATH)

In [ ]:
# ============================================================
# 32. GENERATE TEST PREDICTIONS
# ============================================================

final_model = final_model.to(device)
final_model.eval()

test_feature_dataset = TensorDataset(
    test_features.float()
)

test_feature_loader = DataLoader(
    test_feature_dataset,
    batch_size=best_feature_batch_size,
    shuffle=False
)

test_prediction_indices = []

with torch.inference_mode():
    for (features,) in test_feature_loader:
        features = features.to(device)

        logits = final_model(features)

        predictions = logits.argmax(dim=1)

        test_prediction_indices.extend(
            predictions.cpu().tolist()
        )

predicted_categories = [
    idx_to_class[class_index]
    for class_index in test_prediction_indices
]

submission_df = pd.DataFrame({
    IMAGE_COLUMN: test_identifiers,
    CATEGORY_COLUMN: predicted_categories
})

SUBMISSION_PATH = (
    RESULTS_DIR /
    "test_predictions.csv"
)

submission_df.to_csv(
    SUBMISSION_PATH,
    index=False
)

display(submission_df.head())

print("Test predictions saved to:",
      SUBMISSION_PATH)
print("Number of predictions:",
      len(submission_df))

## Assignment-ready summary

Part I created a reproducible data pipeline for the dress images. The target categories were mapped from text labels to integer class indices using one consistent mapping. The labelled training data was split into training and validation subsets with stratification, preserving approximately the same class proportions in both subsets.

A custom `ImageDressDataset` connects each CSV row to the corresponding image file. Every image is converted to RGB, resized so that its longest side is 224 pixels while maintaining the original aspect ratio, and symmetrically padded to exactly 224 × 224 pixels. The image is then converted to a tensor and normalized using ImageNet statistics. The DataLoaders combine the processed samples into image batches.

For Approach A, the pretrained `dinov2_vits14` model is used as a frozen image encoder. It converts every processed image into a high-dimensional feature vector. Training, validation, and test embeddings are saved so that DINOv2 feature extraction does not need to be repeated during MLP experiments.

The MLP receives the saved DINOv2 embeddings rather than raw image pixels. Adam and cross-entropy loss are used for training. The hyperparameter search first compares learning rates, then architectures, then batch sizes, and finally evaluates a `ReduceLROnPlateau` scheduler. Early stopping monitors validation loss, stops training after a fixed number of non-improving epochs, and restores the model weights from the epoch with the lowest validation loss.

In [ ]:
# ============================================================
# 33. LIST GENERATED OUTPUT FILES
# ============================================================

print("Feature files:")

for path in sorted(FEATURE_DIR.glob("*")):
    print(" -", path)

print("\nResult files:")

for path in sorted(RESULTS_DIR.glob("*")):
    print(" -", path)

print("\nCheckpoint files:")

for path in sorted(CHECKPOINT_DIR.glob("*")):
    print(" -", path)

## Running this notebook non-interactively on the cluster

After the data has been uploaded and extracted, this notebook can be executed with:

```bash
jupyter nbconvert     --to notebook     --execute assignment2_dinov2_mlp_cluster.ipynb     --output assignment2_dinov2_mlp_cluster_executed.ipynb     --ExecutePreprocessor.timeout=-1
```

The executed notebook will retain printed results, tables, and plots. The feature tensors, model checkpoints, CSV summaries, figures, and test predictions are also saved separately in the project directory.